In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.metrics import mean_squared_error

kernel = ConstantKernel(1.0, (1e-4, 1e4)) * \
         RBF(length_scale=1.0, length_scale_bounds=(1e-8, 1e4)) + \
         WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e1))

reg = GaussianProcessRegressor(kernel=kernel)


df = pd.read_csv("components.csv")

X = df.drop(columns=["measurement","no","id"]).values
y = df["measurement"].values


model = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", reg)
])


print(X.shape)
print(X[:5])  # Print first 5 rows of X
print(y.shape)
print(y[:5]) # Print first 5 values of y

In [ ]:
# --- LOOCV Evaluation ---
loo = LeaveOneOut()
preds = []
truth = []
stds = []

for train_idx, test_idx in loo.split(X):
    model.fit(X[train_idx], y[train_idx])
    mean, std = model.predict(X[test_idx], return_std=True)
    preds.append(mean[0])
    stds.append(std[0])
    truth.append(y[test_idx][0])

rmse = np.sqrt(mean_squared_error(truth, preds))
print("LOOCV RMSE:", rmse)

# Optional: print average uncertainty
print("Mean predictive std:", np.mean(stds))


In [ ]:
df2 = pd.read_csv("predict.csv")

X2 = df2.drop(columns=["no","id"]).values

preds = model.predict(X2)    

print("Predicted inrush currents for new data:")
for i, pred in enumerate(preds):
    print(f"Sample {i+1}: {pred}")

